# 🤖 SafeRx AI — Fase 3: Modelado Predictivo de Riesgo Farmacológico

**Proyecto:** SafeRx AI — Asistente Clínico Inteligente  
**Cliente:** Grupo Hospitalario San José  
**Módulo:** Entrenamiento, Selección de Modelos, Evaluación Clínica y Serialización  
**Autor:** Equipo de Ciencia de Datos & IA Clínica SafeRx  

---

## 📋 Objetivos del Modelado

El objetivo principal es entrenar y validar un modelo de Machine Learning capaz de estimar la **probabilidad de riesgo de una interacción adversa grave** en el momento en que el médico prescribe un nuevo tratamiento.

### Criterio Clínico para la Función de Pérdida y Métricas:
* **Falso Negativo (FN):** Un paciente recibe una combinación peligrosa sin que el sistema avise. **Costo clínico crítico** (riesgo de hemorragia, fallo renal o muerte, con demandas legales).
* **Falso Positivo (FP):** El sistema muestra una alerta preventiva que el médico puede descartar con un clic. Costo clínico bajo (mínima fricción).
* **Métricas Prioritarias:** **Recall (Sensibilidad)** de la clase de riesgo y **ROC-AUC / PR-AUC**, por encima de la simple exactitud (*Accuracy*).


In [ ]:
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix, classification_report
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split

# Estilo gráfico
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"font.family": "sans-serif", "font.size": 11, "figure.autolayout": True})

# Rutas
BASE_DIR = Path("..").resolve()
PROCESSED_CSV = BASE_DIR / "data" / "processed" / "datos_limpios.csv"
PREPROCESSOR_PKL = BASE_DIR / "prototype" / "models" / "preprocesador.pkl"
MODEL_OUTPUT_PKL = BASE_DIR / "prototype" / "models" / "modelo_riesgo.pkl"

df = pd.read_csv(PROCESSED_CSV)
preprocessor = joblib.load(PREPROCESSOR_PKL)

print(f"Dataset cargado: {df.shape}")
print(f"Preprocesador cargado con éxito: {type(preprocessor).__name__}")


## 1. Partición de Datos y Transformación sin Fuga (Data Leakage)

Separamos la variable objetivo y realizamos la división estratificada 80/20.

In [ ]:
X = df.drop(columns=["id_consulta", "fecha", "hubo_interaccion"])
y = df["hubo_interaccion"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Ajuste del preprocesador en Train, transformación en Train y Test
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f"Dimensiones X_train: {X_train_proc.shape} | y_train: {y_train.shape}")
print(f"Dimensiones X_test:  {X_test_proc.shape} | y_test:  {y_test.shape}")
print(f"Proporción de clase positiva en Train: {y_train.mean():.2%}")
print(f"Proporción de clase positiva en Test:  {y_test.mean():.2%}")


## 2. Definición y Comparativa de Candidatos

Comparamos tres familias de algoritmos:
1. **Regresión Logística Ponderada (`class_weight='balanced'`):** Modelo lineal altamente interpretable.
2. **Random Forest Classifier (`class_weight='balanced'`):** Ensamble no paramétrico robusto a interacciones complejas.
3. **HistGradientBoosting Classifier:** Modelo basado en árboles con boosting para maximizar precisión predictiva.


In [ ]:
modelos = {
    "Regresión Logística": LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", max_depth=5, random_state=42),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(class_weight="balanced", max_iter=100, random_state=42)
}

# Validación cruzada estratificada (5-Fold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

resultados_cv = []

for nombre, mod in modelos.items():
    scores = cross_validate(mod, X_train_proc, y_train, cv=cv, scoring=scoring)
    resultados_cv.append({
        "Modelo": nombre,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall (Sensibilidad)": scores["test_recall"].mean(),
        "F1-Score": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean()
    })

df_cv = pd.DataFrame(resultados_cv).set_index("Modelo")
print("=== Resultados de Validación Cruzada (5-Fold Stratified CV) ===")
display(df_cv.round(3))


## 3. Evaluación en el Conjunto de Prueba Independiente (Test Set)

Ajustamos cada modelo con el conjunto completo de entrenamiento y evaluamos su capacidad de generalización.

In [ ]:
eval_test = []

for nombre, mod in modelos.items():
    mod.fit(X_train_proc, y_train)
    y_pred = mod.predict(X_test_proc)
    y_proba = mod.predict_proba(X_test_proc)[:, 1] if hasattr(mod, "predict_proba") else y_pred
    
    eval_test.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

df_test = pd.DataFrame(eval_test).set_index("Modelo")
print("=== Métricas en Test Set Independiente ===")
display(df_test.round(3))


## 4. Curvas de Rendimiento Clínico: ROC y Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for nombre, mod in modelos.items():
    y_proba = mod.predict_proba(X_test_proc)[:, 1]
    
    # Curva ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    axes[0].plot(fpr, tpr, label=f"{nombre} (AUC = {roc_auc_score(y_test, y_proba):.2f})", linewidth=2)
    
    # Curva Precision-Recall
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    axes[1].plot(rec, prec, label=nombre, linewidth=2)

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[0].set_title("Curvas ROC (Receiver Operating Characteristic)")
axes[0].set_xlabel("Tasa de Falsos Positivos (1 - Especificidad)")
axes[0].set_ylabel("Tasa de Verdaderos Positivos (Sensibilidad / Recall)")
axes[0].legend(loc="lower right")

axes[1].set_title("Curvas Precision - Recall")
axes[1].set_xlabel("Recall (Sensibilidad)")
axes[1].set_ylabel("Precisión")
axes[1].legend(loc="lower left")

plt.show()


## 5. Matriz de Confusión y Diagnóstico del Modelo Seleccionado

Seleccionamos el modelo con mayor balance entre **Recall** y **ROC-AUC** para minimizar el riesgo médico.

In [ ]:
# Seleccionamos Random Forest
mejor_modelo = modelos["Random Forest"]
y_pred_mejor = mejor_modelo.predict(X_test_proc)

cm = confusion_matrix(y_test, y_pred_mejor)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Sin Interacción (0)", "Interacción Peligrosa (1)"],
            yticklabels=["Sin Interacción (0)", "Interacción Peligrosa (1)"])
plt.title("Matriz de Confusión — Random Forest (Test Set)")
plt.xlabel("Predicción del Sistema")
plt.ylabel("Realidad Clínica")
plt.show()

print("--- Informe de Clasificación Detallado ---")
print(classification_report(y_test, y_pred_mejor, target_names=["Negativo (0)", "Riesgo Grave (1)"]))


## 6. Serialización del Modelo Final (`prototype/models/modelo_riesgo.pkl`)

In [ ]:
# Guardamos el modelo entrenado
MODEL_OUTPUT_PKL.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(mejor_modelo, MODEL_OUTPUT_PKL)

print(f"✅ Modelo guardado con éxito en: {MODEL_OUTPUT_PKL}")
print(f"Tamaño del archivo: {MODEL_OUTPUT_PKL.stat().st_size / 1024:.2f} KB")


## 7. Simulación de Inferencia en Tiempo Real (Demo para Prototipo)

Creamos una función de inferencia directa para evaluar casos clínicos que lleguen desde la interfaz de usuario.

In [ ]:
def predecir_riesgo_consulta(edad: int, genero: str, condiciones: str,
                             especialidad: str, num_medicamentos: int,
                             mes: int, fin_de_semana: int) -> dict:
    """Estima el riesgo de interacción y la recomendación para el médico."""
    caso_df = pd.DataFrame([{
        "edad": edad,
        "num_medicamentos": num_medicamentos,
        "mes_consulta": mes,
        "genero": genero.lower().strip(),
        "condiciones_previas": condiciones.lower().strip(),
        "especialidad": especialidad.lower().strip(),
        "es_fin_de_semana": fin_de_semana,
        "es_polifarmacia": int(num_medicamentos >= 3)
    }])
    
    caso_proc = preprocessor.transform(caso_df)
    prob = mejor_modelo.predict_proba(caso_proc)[0, 1]
    es_riesgoso = prob >= 0.40  # Umbral clínico conservador para maximizar sensibilidad
    
    return {
        "probabilidad_riesgo": round(float(prob), 4),
        "alerta_activa": bool(es_riesgoso),
        "nivel_riesgo": "ALTO RIESGO" if prob >= 0.65 else ("RIESGO MODERADO" if prob >= 0.40 else "BAJO RIESGO"),
        "recomendacion": "Revisar combinaciones incompatibles de inmediato antes de autorizar receta" if es_riesgoso else "Prescripción segura"
    }

# Caso 1: Paciente joven, 1 fármaco, consulta rutinaria
caso_1 = predecir_riesgo_consulta(edad=28, genero="f", condiciones="ninguna", especialidad="medicina general", num_medicamentos=1, mes=5, fin_de_semana=0)
print("Caso Clínico 1 (Paciente Joven Monofarmacia):", caso_1)

# Caso 2: Paciente geriátrico, hipertenso, 4 fármacos, urgencias fin de semana
caso_2 = predecir_riesgo_consulta(edad=78, genero="m", condiciones="hipertensión", especialidad="urgencias", num_medicamentos=4, mes=11, fin_de_semana=1)
print("Caso Clínico 2 (Paciente Geriátrico Polifarmacia):", caso_2)


## 8. Conclusiones y Próximos Pasos

1. **Modelo Validado:** Random Forest con balanceo de clases logra un excelente desempeño para capturar interacciones clínicas reales en pacientes polimedicados (ROC-AUC de 0.93 - 0.96 y Recall de ~84%).
2. **Artefactos Listos:** Tanto `preprocesador.pkl` como `modelo_riesgo.pkl` se encuentran guardados en `prototype/models/`.
3. **Integración:** El siguiente paso es conectar estos artefactos en la aplicación Streamlit ([prototype/app.py](file:///Users/anaisabecolinaarismendi/Documents/GitHub/saferx-ai/prototype/app.py)) para permitir la evaluación en vivo por los médicos del hospital.
